# SSL + Rebalance Fine-tuning with Adaptive Class-Weighting

**Purpose:** responds to correction #3 (new mathematical formulation) *and* closes a loose
end from the original supervisor review, which flagged that the framework is called
"adaptive rebalancing" even though the focal-loss class weights (MEL=4, BKL=2, NV=1) are
fixed constants chosen by hand, not actually adaptive.

**The formula.** After a short warm-up period (epochs 1-3, to avoid reacting to noisy early
validation estimates), the MEL class weight is updated after every epoch by a simple
proportional control rule:

    w_MEL(t+1) = clip( w_MEL(t) + eta * sign(R_target - R_MEL(t)) , w_min, w_max )

where `R_MEL(t)` is the validation MEL recall observed after epoch t, `R_target` is a target
recall (set below the level that would force a degenerate all-MEL classifier), `eta` is a
fixed step size, and clipping keeps the weight in a sane range so it cannot destroy BKL/NV
performance. In words: if MEL recall is below target, push the weight up a step; if MEL
recall has already reached target (or precision is visibly collapsing), ease off. BKL and NV
weights stay fixed at their original values (2.0 and 1.0) -- only the MEL weight adapts,
since MEL recall is the paper's stated primary endpoint.

This is a small, legitimate, literally-adaptive mechanism -- not a claim of a major new
theoretical contribution. It is reported as such in the paper: an incremental refinement of
the rebalancing scheme, evaluated with the same 3-seed protocol and the same
validation-macro-F1 checkpoint selection as everything else in the paper.

Loads `ssl_encoder_real_24000.pth` directly; does not repeat SSL pretraining.


In [23]:
import os
import sys
import glob
import json
import random
from collections import Counter

# Core Data & Processing
import pandas as pd
import numpy as np
from PIL import Image

# PyTorch Ecosystem
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Dynamically locate config.py
config_matches = glob.glob('/kaggle/input/**/config.py', recursive=True)
if not config_matches:
    raise FileNotFoundError("Could not find config.py in /kaggle/input")
config_path = config_matches[0]

spec = importlib.util.spec_from_file_location("config", config_path)
config = importlib.util.module_from_spec(spec)
sys.modules["config"] = config
spec.loader.exec_module(config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
print(f"Device: {device}") 

# Dynamically search for the folder containing expA_train.csv
train_csv_matches = glob.glob('/kaggle/input/**/expA_train.csv', recursive=True)
if not train_csv_matches:
    raise FileNotFoundError("Could not locate expA_train.csv in /kaggle/input")

BACKUP_DIR = os.path.dirname(train_csv_matches[0])
print(f"Auto-detected CSV Directory: {BACKUP_DIR}")

# Dynamically search for the labeled images folder
labeled_dir_matches = glob.glob('/kaggle/input/**/ISIC_2019_Training_Input', recursive=True)
LABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'

# Dynamically search for SSL encoder weights
ssl_matches = glob.glob('/kaggle/input/**/ssl_encoder_real_24000.pth', recursive=True)
SSL_ENCODER_PATH = ssl_matches[0] if ssl_matches else ''

assert os.path.exists(SSL_ENCODER_PATH), "ssl_encoder_real_24000.pth not found."

# Output directory where model checkpoints and results will be saved
OUTPUT_DIR = '/kaggle/working'

# --- Adaptive weighting hyperparameters --- 
WARMUP_EPOCHS = 3
TARGET_MEL_RECALL = 0.45 
ETA = 0.5 
W_MIN, W_MAX = 2.0, 8.0

Device: cuda
Auto-detected CSV Directory: /kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs


In [24]:
# =====================================================================
# DATA -- identical to the original fine-tuning notebook (no synthetic data here;
# this notebook isolates the effect of adaptive weighting on its own).
# =====================================================================
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, mel_multiplier=5, transform_normal=None, transform_mel=None, seed=42):
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel
        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True) \
                             .sample(frac=1, random_state=seed).reset_index(drop=True)
    def __len__(self):
        return len(self.expanded_df)
    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        
        img_name = str(row['image'])
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'
            
        # Check direct path first
        img_path = os.path.join(LABELED_DIR, img_name)
        
        # If not directly in LABELED_DIR, check one subfolder down
        if not os.path.exists(img_path):
            sub_matches = glob.glob(f"{LABELED_DIR}/**/{img_name}", recursive=True)
            if sub_matches:
                img_path = sub_matches[0]
            else:
                raise FileNotFoundError(f"Could not find image {img_name} anywhere in {LABELED_DIR}")

        img = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['label']]
        
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
            
        return img, label


In [25]:
# =====================================================================
# MODEL DEFINITIONS -- identical to the original notebook
# =====================================================================
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class FocalLoss(nn.Module):
    """Same focal loss as before, but alpha (class weights) is passed in fresh each
    epoch from outside, since it now changes over training."""
    def __init__(self, gamma=1.5):
        super().__init__()
        self.gamma = gamma
    def forward(self, inputs, targets, alpha):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

def train_epoch(model, loader, optimizer, criterion, alpha):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels, alpha)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, classes):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(all_labels, all_preds, target_names=classes,
                                    output_dict=True, zero_division=0)
    mel_recall = report.get('MEL', {}).get('recall', 0.0)
    return acc, macro_f1, mel_recall, all_labels, all_preds


In [26]:
# =====================================================================
# 3-SEED FINE-TUNING WITH ADAPTIVE MEL WEIGHTING
# =====================================================================
BASE_WEIGHTS = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}  # MEL=4.0 is also the starting point here
seeds = [42, 123, 2024]
all_results = {}

for seed in seeds:
    print(f"\n{'#'*40}\nSEED {seed}\n{'#'*40}")
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_ds = AugmentedDataset(train_df, 5, normal_transform, mel_transform, seed=seed)
    val_ds = AugmentedDataset(val_df, 1, test_transform, test_transform, seed=seed)
    test_ds = AugmentedDataset(test_df, 1, test_transform, test_transform, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)
    test_loader = DataLoader(test_ds, batch_size=16)

    encoder = SimpleEncoder()
    encoder.load_state_dict(torch.load(SSL_ENCODER_PATH, map_location=device))
    for p in encoder.parameters():
        p.requires_grad = True
    model = SSLClassifier(encoder, len(train_ds.classes)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = FocalLoss(gamma=1.5)

    mel_weight = BASE_WEIGHTS['MEL']  # this is the value the adaptive rule will move
    weight_history = [mel_weight]

    epochs, patience = 30, 7
    best_val_macro_f1, epochs_no_improve, best_epoch = -1.0, 0, -1
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": [],
               "val_mel_recall": [], "mel_weight": []}
    seed_ckpt = os.path.join(OUTPUT_DIR, f'adaptive_best_seed_{seed}.pth')

    for epoch in range(epochs):
        alpha = torch.tensor(
            [mel_weight if c == 'MEL' else BASE_WEIGHTS[c] for c in train_ds.classes],
            dtype=torch.float32
        ).to(device)

        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, alpha)
        val_acc, val_macro_f1, val_mel_recall, _, _ = evaluate(model, val_loader, train_ds.classes)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_macro_f1)
        history["val_mel_recall"].append(val_mel_recall)
        history["mel_weight"].append(mel_weight)

        print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}, "
              f"Val Macro-F1={val_macro_f1:.3f}, Val MEL Recall={val_mel_recall:.3f}, "
              f"MEL weight={mel_weight:.2f}")

        # --- Adaptive update rule (see markdown above for the formula) ---
        if epoch + 1 > WARMUP_EPOCHS:
            direction = 1.0 if val_mel_recall < TARGET_MEL_RECALL else -1.0
            mel_weight = float(np.clip(mel_weight + ETA * direction, W_MIN, W_MAX))
        weight_history.append(mel_weight)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1, best_epoch = val_macro_f1, epoch + 1
            torch.save(model.state_dict(), seed_ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(seed_ckpt, map_location=device))
    test_acc, test_macro_f1, test_mel_recall, true_labels, pred_labels = evaluate(
        model, test_loader, train_ds.classes)
    report = classification_report(true_labels, pred_labels, target_names=train_ds.classes,
                                    output_dict=True, zero_division=0)

    print(f"\n>>> Seed {seed}: Test Acc={test_acc:.3f}, Test Macro-F1={test_macro_f1:.3f}, "
          f"MEL Recall={test_mel_recall:.1%} <<<")
    print(f"Prediction distribution: {Counter(pred_labels)}")

    all_results[seed] = {
        "test_accuracy": test_acc, "test_macro_f1": test_macro_f1, "mel_recall": test_mel_recall,
        "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1,
        "report": report, "history": history, "final_mel_weight": mel_weight,
    }

accs = [all_results[s]["test_accuracy"] for s in seeds]
f1s = [all_results[s]["test_macro_f1"] for s in seeds]
recalls = [all_results[s]["mel_recall"] for s in seeds]

print("\n" + "="*40)
print("3-SEED SUMMARY -- SSL + Adaptive-Weight Rebalance")
print("="*40)
print(f"Mean Test Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Mean Test Macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"Mean MEL Recall:    {np.mean(recalls):.1%} +/- {np.std(recalls):.1%}")
print("\nCompare against the fixed-weight full framework: 41.7% +/- 5.9% MEL recall")

with open(os.path.join(OUTPUT_DIR, 'ssl_adaptive_weight_3seed_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print("\nSaved results to ssl_adaptive_weight_3seed_results.json")



########################################
SEED 42
########################################
Epoch 1: Train Acc=0.276, Val Acc=0.173, Val Macro-F1=0.131, Val MEL Recall=1.000, MEL weight=4.00
Epoch 2: Train Acc=0.396, Val Acc=0.471, Val Macro-F1=0.263, Val MEL Recall=0.429, MEL weight=4.00
Epoch 3: Train Acc=0.522, Val Acc=0.500, Val Macro-F1=0.264, Val MEL Recall=0.286, MEL weight=4.00
Epoch 4: Train Acc=0.573, Val Acc=0.558, Val Macro-F1=0.312, Val MEL Recall=0.286, MEL weight=4.00
Epoch 5: Train Acc=0.621, Val Acc=0.606, Val Macro-F1=0.376, Val MEL Recall=0.286, MEL weight=4.50
Epoch 6: Train Acc=0.624, Val Acc=0.587, Val Macro-F1=0.382, Val MEL Recall=0.286, MEL weight=5.00
Epoch 7: Train Acc=0.658, Val Acc=0.625, Val Macro-F1=0.427, Val MEL Recall=0.286, MEL weight=5.50
Epoch 8: Train Acc=0.639, Val Acc=0.615, Val Macro-F1=0.421, Val MEL Recall=0.286, MEL weight=6.00
Epoch 9: Train Acc=0.650, Val Acc=0.625, Val Macro-F1=0.427, Val MEL Recall=0.286, MEL weight=6.50
Epoch 10: Train Ac